In [ ]:
import requests
import re
import json
from Bio import SeqIO
import subprocess
import sys

def http_function(endpoint, **kwargs):
    r = requests.get(endpoint, **kwargs)

    if r.status_code != 200:
        return {"error": r.text}

    return r.json()

class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name

    def _get_uniprot(self, accession):
        # use HW2_1
        endpoint = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
        http_args = {}
        return http_function(endpoint, **http_args)

    def _get_ensembl(self, id):
        # use HW2_1
        endpoint = f"https://rest.ensembl.org/lookup/id/{id}"
        http_args = {
            "headers": {"Content-Type": "application/json"}
          }
        return http_function(endpoint, **http_args)

    def _uniprot_parse_response(self, resp: dict):
        # use HW2_1
        try:
            accession = resp.get("primaryAccession")
        except Exception:
            accession = None
    
        organism = resp.get("organism", {}).get("scientificName")
        geneInfo = resp.get("genes", [])
        sequenceInfo = resp.get("sequence", {})
        type_ = resp.get("entryType")

        output = {
            accession: {
                "organism": organism,
                "geneInfo": geneInfo,
                "sequenceInfo": sequenceInfo,
                "type": type_
            }
        }
        return output

    def _ensembl_parse_response(self, resp: dict):
        # use HW2_1
        try:
            id = resp["id"]
        except KeyError:
            id = None

        object_type = resp.get("object_type")
        assembly_name = resp.get("assembly_name")
        species = resp.get("species")
        db_type = resp.get("db_type")
        biotype = resp.get("biotype")
        display_name = resp.get("display_name")
        id = resp.get("id")
        description = resp.get("description")
        canonical_transcript = resp.get("canonical_transcript")
        source = resp.get("source")
  
        output = {
            id: {
                "object_type": object_type,
                "species": species,
                "assembly_name": assembly_name,
                "biotype": biotype,
                "display_name": display_name,
                "id": id,
                "db_type": db_type,
                "description": description,
                "source": source,
                "canonical_transcript": canonical_transcript
            }
        }
        return output

    def _access_database(self, id, database, seq_description, seq_sequence):
        # this function calls either _get_uniprot() or _get_ensembl()
        # an then _uniprot_parse_response() or _ensembl_parse_response()
        # outputs a dictionary with results (see example in test_data)
        output = {}

        if database == "uniprot":

            resp = self._get_uniprot(id)
            parsed = self._uniprot_parse_response(resp)

        elif database == "ensembl":

            resp = self._get_ensembl(id)
            parsed = self._ensembl_parse_response(resp)

        output = {
            f"file_info_{id}": {
                "description": seq_description,
                "sequence": str(seq_sequence)
            },
            f"database_info_{id}": parsed[id]
        }

        return output

    def seqkit_stats(self):
        # this function calls seqkit via subprocess
        # if error arises, returns stderr and finishes the parser execution
        # if stats are collected, return the result (see example in test_data)
        
        try:
            result = subprocess.run(
                ["seqkit", "stats", self.filename],
                capture_output=True,
                text=True,
                check=True
            )

        except subprocess.CalledProcessError as e:
            return {"ERROR": e.stderr}

        lines = result.stdout.strip().split("\n")

        header = lines[0].split()
        values = lines[1].split()

        stats = dict(zip(header[1:], values[1:]))

        fasta_type = stats.get("type")
        fasta_num_seqs = int(stats.get("num_seqs"))

        seqkit_result = {
            "fasta_seqkit_stat_info": stats,
            "fasta_type": fasta_type,
            "fasta_num_seqs": fasta_num_seqs
        }
    
        return seqkit_result

    def biopython_parser(self, seqkit_result):
        # accepts seqkit_result dict
        # depending on fasta file type selects regular expression
        # parses fasta file via biopython
        # iterates over sequences and calls relevant database via _access_databases()
        # returns final result (info about each sequence from FASTA file + info from database)
        
        output = {}

        fasta_type = seqkit_result["fasta_type"]

        if fasta_type == "Protein":

            database = "uniprot"

            # UniProt accession regex
            regex = re.compile(r"[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]{5}")

        else:

            database = "ensembl"

            # Ensembl ID regex
            regex = re.compile(r"ENS[A-Z]*\d+")

        output["DB_name"] = database

        for record in SeqIO.parse(self.filename, "fasta"):

            desc = record.description
            seq = record.seq

            match = regex.search(desc)

            if match:

                id = match.group()

                db_info = self._access_database(id, database, desc, seq)

                output.update(db_info)

            else:

                output["WARNING"] = {"No ID match found."}

        return output

    def show_output(self, output, indent=0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))


parser = MyFastaParser("test_file.fasta")

stats = parser.seqkit_stats()

biopython = parser.biopython_parser(stats)

parser.show_output(biopython)